In [2]:
# %% [markdown]
# BEACON — Post-preprocessing validation (run once before training)
# This is a sanity check, not a re-run of EDA — it verifies Steps 5-10
# didn't silently introduce problems, not that the raw data is clean
# (you've already confirmed that).

# %%
import os
import numpy as np
import pandas as pd

FINAL_DIR = r"D:\Malware Dataset\processed_final"

net_train = pd.read_parquet(os.path.join(FINAL_DIR, "net_train.parquet"))
net_test = pd.read_parquet(os.path.join(FINAL_DIR, "net_test.parquet"))
mem_train = pd.read_parquet(os.path.join(FINAL_DIR, "mem_train.parquet"))
mem_test = pd.read_parquet(os.path.join(FINAL_DIR, "mem_test.parquet"))

net_weights = np.load(os.path.join(FINAL_DIR, "net_sample_weights.npy"))
mem_weights = np.load(os.path.join(FINAL_DIR, "mem_sample_weights.npy"))

# %%
# Check 1: column parity between train and test (post correlation-pruning)
def check_columns_match(train, test, name):
    train_cols = set(train.columns)
    test_cols = set(test.columns)
    if train_cols == test_cols:
        print(f"[OK] {name}: train/test columns match ({len(train_cols)} columns)")
    else:
        print(f"[MISMATCH] {name}:")
        print(f"  In train only: {train_cols - test_cols}")
        print(f"  In test only:  {test_cols - train_cols}")

check_columns_match(net_train, net_test, "Network")
check_columns_match(mem_train, mem_test, "Memory")

# %%
# Check 2: NaN / Inf introduced anywhere numeric
def check_nan_inf(df, name):
    numeric_df = df.select_dtypes(include="number")
    nan_counts = numeric_df.isnull().sum()
    inf_counts = np.isinf(numeric_df).sum()

    bad_nan = nan_counts[nan_counts > 0]
    bad_inf = inf_counts[inf_counts > 0]

    if len(bad_nan) == 0 and len(bad_inf) == 0:
        print(f"[OK] {name}: no NaN or Inf found")
    else:
        if len(bad_nan):
            print(f"[NaN FOUND] {name}:\n{bad_nan}")
        if len(bad_inf):
            print(f"[INF FOUND] {name}:\n{bad_inf}")

for df, name in [(net_train, "Network train"), (net_test, "Network test"),
                  (mem_train, "Memory train"), (mem_test, "Memory test")]:
    check_nan_inf(df, name)

# %%
# Check 3: scaled columns actually land within [0, 1]
# (a value outside this range on TEST is expected occasionally — it just
# means a test-set value fell outside train's observed min/max — but a
# widespread violation signals something went wrong)
def check_scaling_range(df, name, tolerance=1e-6):
    numeric_df = df.select_dtypes(include="number")
    out_of_range = {}
    for col in numeric_df.columns:
        col_min, col_max = numeric_df[col].min(), numeric_df[col].max()
        if col_min < -tolerance or col_max > 1 + tolerance:
            out_of_range[col] = (col_min, col_max)

    if not out_of_range:
        print(f"[OK] {name}: all numeric columns within [0, 1]")
    else:
        print(f"[OUT OF RANGE] {name} — {len(out_of_range)} column(s):")
        for col, (mn, mx) in list(out_of_range.items())[:10]:  # show first 10
            print(f"    {col}: min={mn:.3f}, max={mx:.3f}")

check_scaling_range(net_train, "Network train")
check_scaling_range(net_test, "Network test")  # some spread beyond [0,1] here is OK
check_scaling_range(mem_train, "Memory train")
check_scaling_range(mem_test, "Memory test")

# %%
# Check 4: sample weights align in length with their training sets
assert len(net_weights) == len(net_train), \
    f"Network weight/row mismatch: {len(net_weights)} weights vs {len(net_train)} rows"
assert len(mem_weights) == len(mem_train), \
    f"Memory weight/row mismatch: {len(mem_weights)} weights vs {len(mem_train)} rows"
print("[OK] Sample weight arrays match their training set row counts")

# %%
# Check 5: quick recap of final shapes and label distributions
print("\n" + "=" * 60)
print("FINAL SHAPES")
print("=" * 60)
print(f"Network — train: {net_train.shape}, test: {net_test.shape}")
print(f"Memory  — train: {mem_train.shape}, test: {mem_test.shape}")

print("\nNetwork train label distribution:")
print(net_train["label"].value_counts())
print("\nMemory train label distribution (Exploit should show is_synthetic split):")
print(mem_train["label"].value_counts())
if "is_synthetic" in mem_train.columns:
    print("\nExploit synthetic vs real breakdown:")
    print(mem_train[mem_train["label"] == "Exploit"]["is_synthetic"].value_counts())

print("\nIf all checks above show [OK], you're clear to move to model training.")

[OK] Network: train/test columns match (188 columns)
[MISMATCH] Memory:
  In train only: {'is_synthetic'}
  In test only:  set()
[OK] Network train: no NaN or Inf found
[OK] Network test: no NaN or Inf found
[OK] Memory train: no NaN or Inf found
[OK] Memory test: no NaN or Inf found
[OK] Network train: all numeric columns within [0, 1]
[OUT OF RANGE] Network test — 15 column(s):
    mean_header_bytes: min=0.000, max=1.181
    fwd_variance_header_bytes: min=0.000, max=1.003
    fwd_segment_size_cov: min=0.000, max=1.073
    bytes_rate: min=0.000, max=1.013
    fwd_bytes_rate: min=0.000, max=1.177
    packets_rate: min=0.000, max=1.030
    fwd_packets_rate: min=0.000, max=1.444
    down_up_rate: min=0.000, max=1.007
    avg_bwd_bulk_rate: min=0.000, max=1.379
    fwd_bulk_duration: min=0.000, max=1.138
[OK] Memory train: all numeric columns within [0, 1]
[OUT OF RANGE] Memory test — 12 column(s):
    pslist.nproc: min=-0.003, max=0.990
    pslist.avg_threads: min=-0.094, max=0.989
    d